# 02 · Information extraction

The free local LLM (Ollama) reads a chapter and returns the story entities, facts and events in it, as structured data. Each fact keeps the exact sentence it came from, so a contradiction flagged later can always cite its source.

**Before running:** the Ollama app must be running, and the kernel (top right) should say **Continuity Editor (.venv)**. Run the cells in order, starting with Setup.

## 0. Setup

In [ ]:
import importlib.util
import sys

print(f"Python {sys.version.split()[0]} at {sys.executable}")
if sys.version_info < (3, 11):
    raise RuntimeError("This kernel's Python is too old: click the kernel name (top right) and choose 'Continuity Editor (.venv)'.")

# Libraries this notebook uses; continuity_editor is this project's own code.
REQUIRED = ["pandas", "ollama", "rapidfuzz", "pydantic", "pydantic_settings", "continuity_editor"]
missing = [name for name in REQUIRED if importlib.util.find_spec(name) is None]
if missing:
    print("Installing missing libraries:", ", ".join(missing))
    %pip install --quiet -r ../requirements.txt -e ..
    print("Done. Restart the kernel (Restart in the toolbar), then run the notebook again.")
else:
    print("All libraries are installed.")

## 1. The book, split into chapters

`scripts/prepare_novel.py` downloaded *The Wonderful Wizard of Oz* from Project Gutenberg, removed the licence text and table of contents, and split it into chapters. Long chapters are sent to the model in chunks of about 900 words.

In [ ]:
import pandas as pd

from continuity_editor.extraction import chunk_text, extract_novel, load_chapters
from continuity_editor.llm import LLMClient
from continuity_editor.paths import chapters_path, extraction_cache_dir

NOVEL = "oz"
if not chapters_path(NOVEL).exists():
    raise FileNotFoundError("Run `python scripts/prepare_novel.py` in the terminal first.")
chapters = load_chapters(chapters_path(NOVEL))
pd.DataFrame(
    [{"chapter": c.number, "title": c.title, "words": c.word_count, "chunks": len(chunk_text(c.text))} for c in chapters]
).set_index("chapter")

## 2. Extract one chapter

Each chunk goes through two passes:
1. **Entities:** the characters, places and world elements in the chunk.
2. **Facts and events:** only about the entities from pass 1 (the model can't name anyone else), each with a quoted sentence as evidence.

Any fact whose quote can't be found in the chapter is dropped. The first run of a chapter takes a few minutes on an 8 GB laptop; the result is saved, so running the cell again is instant.

In [ ]:
CHAPTER = 1

llm = LLMClient()
llm.verify()
result = extract_novel(llm, NOVEL, [chapters[CHAPTER - 1]], extraction_cache_dir())[0]
print(
    f"Chapter {result.chapter_number} ({result.chapter_title}): {len(result.entities)} entities, "
    f"{len(result.facts)} facts, {len(result.events)} events. "
    f"Dropped {result.unverified_dropped} with unverifiable quotes. Took {result.seconds:.0f}s with {result.model}."
)

In [ ]:
pd.DataFrame([e.model_dump() for e in result.entities])

In [ ]:
pd.set_option("display.max_colwidth", 120)
pd.DataFrame([f.model_dump() for f in result.facts])

In [ ]:
pd.DataFrame([e.model_dump() for e in result.events])

## 3. The whole book

To extract every chapter, run `python scripts/extract_novel.py` in the terminal. It saves each chapter as it finishes, so it can be stopped and restarted without losing work. The results feed Step 3, which stores them in the Neo4j story bible.